# Manager input and market-engine validation

This notebook independently summarizes the primary and holdout experiment outputs produced by `scripts/test-manager-input-matrix.ts`. Corrected availability rows replace the original rows whose moved-by-60 numerator and denominator used different eligibility rules.

In [1]:
import json, math
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
analysis = root / 'analysis'

def load(name):
    return json.loads((analysis / name).read_text())

primary = load('manager-input-matrix-primary.json')
holdout = load('manager-input-matrix-holdout.json')
primary_correction = load('manager-input-matrix-primary-correction.json')
holdout_correction = load('manager-input-matrix-holdout-correction.json')

def corrected_rows(full, correction):
    replacements = {row['id']: row for row in correction['scenarios']}
    return [replacements.get(row['id'], row) for row in full['scenarios']]

p_rows = corrected_rows(primary, primary_correction)
h_rows = corrected_rows(holdout, holdout_correction)
p = {row['id']: row for row in p_rows}
h = {row['id']: row for row in h_rows}
len(p_rows), len(h_rows), primary['method']['totalSimulatedServices'] + holdout['method']['totalSimulatedServices']

(32, 32, 77550)

In [2]:
def z_difference(left, right, metric):
    a, b = left['metrics'][metric], right['metrics'][metric]
    se = math.sqrt(a['standardDeviation'] ** 2 / left['runs'] + b['standardDeviation'] ** 2 / right['runs'])
    return 0 if se == 0 else (a['mean'] - b['mean']) / se

agreement = []
for scenario_id in p:
    for metric in ('actualRevenueRatioPct', 'averageProductRangePct'):
        agreement.append((scenario_id, metric, z_difference(p[scenario_id], h[scenario_id], metric)))

outside_95 = [row for row in agreement if abs(row[2]) > 1.96]
max_difference = max(agreement, key=lambda row: abs(row[2]))
safe_ids = [scenario_id for scenario_id in p if scenario_id not in {'availability-unmapped', 'availability-combined'}]
safe_services = sum(p[i]['runs'] + h[i]['runs'] for i in safe_ids)
safe_failures = sum(p[i]['invariants']['totalFailures'] + h[i]['invariants']['totalFailures'] for i in safe_ids)
unmapped_changes = sum(p[i]['invariants']['ineligiblePriceChanges'] + h[i]['invariants']['ineligiblePriceChanges'] for i in ('availability-unmapped', 'availability-combined'))
unmapped_decisions = (10 * 72 * (2000 + 500)) + (4 * 72 * (2000 + 500))

{
    'safe_services': safe_services,
    'safe_failures': safe_failures,
    'zero_failure_upper_95_pct': 300 / safe_services,
    'unmapped_invalid_changes': unmapped_changes,
    'unmapped_possible_decisions': unmapped_decisions,
    'unmapped_invalid_change_rate_pct': 100 * unmapped_changes / unmapped_decisions,
    'agreement_comparisons': len(agreement),
    'outside_nominal_95_count': len(outside_95),
    'largest_absolute_z': max_difference,
}

{'safe_services': 72550,
 'safe_failures': 0,
 'zero_failure_upper_95_pct': 0.004135079255685734,
 'unmapped_invalid_changes': 2313693,
 'unmapped_possible_decisions': 2520000,
 'unmapped_invalid_change_rate_pct': 91.81321428571428,
 'agreement_comparisons': 64,
 'outside_nominal_95_count': 3,
 'largest_absolute_z': ('demand-runaway',
  'averageProductRangePct',
  -2.3192070546868004)}

In [3]:
headline_ids = ['turnover-micro', 'turnover-quiet', 'baseline', 'turnover-busy', 'turnover-extreme']
turnover_table = [{
    'scenario': p[i]['label'],
    'target_gbp': p[i]['expectedRevenueGbp'],
    'actual_vs_target_pct': p[i]['metrics']['actualRevenueRatioPct']['mean'],
    'revenue_ci95_margin': p[i]['metrics']['actualRevenueRatioPct']['ci95Margin'],
    'night_range_pct': p[i]['metrics']['averageProductRangePct']['mean'],
    'moving_rounds_pct': p[i]['metrics']['movingDecisionRatePct']['mean'],
} for i in headline_ids]
turnover_table

[{'scenario': '£500 expected takings',
  'target_gbp': 500,
  'actual_vs_target_pct': 108.159,
  'revenue_ci95_margin': 1.8653,
  'night_range_pct': 1.0027,
  'moving_rounds_pct': 19.1881},
 {'scenario': '£2,500 expected takings',
  'target_gbp': 2500,
  'actual_vs_target_pct': 106.5159,
  'revenue_ci95_margin': 0.7668,
  'night_range_pct': 4.0141,
  'moving_rounds_pct': 68.5539},
 {'scenario': 'Standard manager setup',
  'target_gbp': 10000,
  'actual_vs_target_pct': 105.8876,
  'revenue_ci95_margin': 0.3911,
  'night_range_pct': 9.6996,
  'moving_rounds_pct': 94.0055},
 {'scenario': '£25,000 expected takings',
  'target_gbp': 25000,
  'actual_vs_target_pct': 104.9968,
  'revenue_ci95_margin': 0.2485,
  'night_range_pct': 12.6514,
  'moving_rounds_pct': 97.0954},
 {'scenario': '£50,000 expected takings',
  'target_gbp': 50000,
  'actual_vs_target_pct': 103.621,
  'revenue_ci95_margin': 0.1704,
  'night_range_pct': 13.6389,
  'moving_rounds_pct': 98.0524}]

In [4]:
comparison_ids = ['range-tight', 'range-narrow', 'baseline', 'range-wide', 'live-1', 'live-2', 'live-5', 'live-10', 'crash-one-5', 'crash-one-10', 'crash-four']
comparison_table = [{
    'scenario': p[i]['label'],
    'night_range_pct': p[i]['metrics']['averageProductRangePct']['mean'],
    'average_round_move_pct': p[i]['metrics']['averageRoundMovePct']['mean'],
    'moving_rounds_pct': p[i]['metrics']['movingDecisionRatePct']['mean'],
    'near_limit_pct': p[i]['metrics']['nearLimitDecisionRatePct']['mean'],
} for i in comparison_ids]
comparison_table

[{'scenario': 'Symmetric ±5% price range',
  'night_range_pct': 2.4395,
  'average_round_move_pct': 0.1809,
  'moving_rounds_pct': 82.6493,
  'near_limit_pct': 0},
 {'scenario': 'Symmetric ±10% price range',
  'night_range_pct': 4.8581,
  'average_round_move_pct': 0.3565,
  'moving_rounds_pct': 89.8322,
  'near_limit_pct': 0},
 {'scenario': 'Standard manager setup',
  'night_range_pct': 9.6996,
  'average_round_move_pct': 0.7209,
  'moving_rounds_pct': 94.0055,
  'near_limit_pct': 0},
 {'scenario': 'Symmetric ±40% price range',
  'night_range_pct': 18.9572,
  'average_round_move_pct': 1.4524,
  'moving_rounds_pct': 96.0727,
  'near_limit_pct': 0},
 {'scenario': '1 live drink per category',
  'night_range_pct': 0,
  'average_round_move_pct': 0,
  'moving_rounds_pct': 0,
  'near_limit_pct': 0},
 {'scenario': '2 live drinks per category',
  'night_range_pct': 15.2905,
  'average_round_move_pct': 1.1405,
  'moving_rounds_pct': 94.8938,
  'near_limit_pct': 0},
 {'scenario': '5 live drinks p

In [5]:
configured_expected_basket_units = 2.55
group_size_weights = [(1, .14), (2, .38), (3, .21), (4, .16), (5, .07), (6, .04)]
weighted_group_size = sum(size * weight for size, weight in group_size_weights)
{
    'configured_expected_basket_units': configured_expected_basket_units,
    'weighted_group_size_from_same_model': weighted_group_size,
    'relative_mismatch_pct': 100 * (weighted_group_size / configured_expected_basket_units - 1),
    'baseline_revenue_bias_pct': p['baseline']['metrics']['actualRevenueRatioPct']['mean'] - 100,
    'baseline_bias_ci95': [p['baseline']['metrics']['actualRevenueRatioPct']['ci95Low'] - 100, p['baseline']['metrics']['actualRevenueRatioPct']['ci95High'] - 100],
}

{'configured_expected_basket_units': 2.55,
 'weighted_group_size_from_same_model': 2.7600000000000002,
 'relative_mismatch_pct': 8.235294117647074,
 'baseline_revenue_bias_pct': 5.887600000000006,
 'baseline_bias_ci95': [5.496399999999994, 6.278700000000001]}

## Interpretation guardrails

The simulation establishes calculation behaviour under the encoded customer model. It does not validate that the customer priors match a particular real venue. Publication fault checks are source-path checks; fully proving POS behaviour still requires a transactional integration test against a connector sandbox.